<!-- TASK_A_REWRITTEN_ANALYSIS -->
# Dyck Syn-to-Rea: Task A Length and Noise

这个 notebook 记录 Task A 的第一轮 Transformer baseline：在当前 Dyck-1 随机生成器上，系统改变序列长度、Dyck token 稀疏度和 noise vocabulary，观察 Transformer 的 next-token 行为、hidden-state counter 表征，以及这些表征是否真的进入输出头。

读法上分三层：

1. 先看 Transformer runs 是否训练、抽 hidden、跑 probe 都完整。
2. 再看 behavior accuracy 和 hidden counter probe 之间的差距。
3. 最后用六个 follow-up probes 拆开这个差距到底来自哪里。


In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import yaml

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent

CONFIG_TEMPLATE = ROOT / 'configs' / 'dyck_counter_length_noise_transformer.yaml'
GENERATED_CONFIG_DIR = ROOT / 'configs' / 'generated_task_a'
RESULTS_ROOT = ROOT / 'results'

WRITE_CONFIGS = False
RUN_TRAINING = False
DEVICE = "cuda"  
NUM_EXAMPLES = 4096
EXTRACT_BATCH_SIZE = 512
PROBE_MAX_ROWS = 30000
PROBE_MAX_CLASSES = 200

ROOT

## Experiment Settings

所有设置都使用同一个 3-layer Transformer、seed=0、final checkpoint hidden states，并抽取 all layers / all positions。变化的只有 Dyck 总长度、序列长度、噪声密度和 noise vocabulary。

当前 Dyck sampler 是 stochastic Markov-style balanced path：当 height=0、open 数用完、或剩余步数必须全 close 时，下一步由规则强制决定；其余 free step 近似 50/50 open-vs-close。这个生成机制非常重要，因为它给 next-token accuracy 设置了一个 Bayes-optimal 上限。


In [ ]:
TASK_A_GROUPS = [
    dict(name='tiny_extreme_long', dyck_pairs=10, total_length=20, seq_len=2000, repeat_prob=0.01, num_noise_tokens=64, seeds=[0], batch_size=4),
    dict(name='clean_short', dyck_pairs=24, total_length=48, seq_len=48, repeat_prob=1.0, num_noise_tokens=4, seeds=[0], batch_size=128),
    dict(name='noisy_short', dyck_pairs=24, total_length=48, seq_len=120, repeat_prob=0.5, num_noise_tokens=16, seeds=[0], batch_size=128),
    dict(name='sparse_medium', dyck_pairs=100, total_length=200, seq_len=400, repeat_prob=0.25, num_noise_tokens=16, seeds=[0], batch_size=128),
    dict(name='sparse_long', dyck_pairs=200, total_length=400, seq_len=1000, repeat_prob=0.25, num_noise_tokens=16, seeds=[0], batch_size=16),
    dict(name='extreme_long', dyck_pairs=200, total_length=400, seq_len=2000, repeat_prob=0.1, num_noise_tokens=64, seeds=[0], batch_size=4),
]

pd.DataFrame(TASK_A_GROUPS)


In [ ]:
def make_group_config(group: dict) -> dict:
    cfg = yaml.safe_load(CONFIG_TEMPLATE.read_text())
    cfg['experiment']['name'] = f"dyck_counter_task_a_{group['name']}"
    cfg['experiment']['seeds'] = list(group['seeds'])
    cfg['task'].update(
        dyck_pairs=group['dyck_pairs'],
        total_length=group['total_length'],
        seq_len=group['seq_len'],
        repeat_prob=group['repeat_prob'],
        num_noise_tokens=group['num_noise_tokens'],
        prefix_probe_max_len=group['total_length'],
    )
    cfg['training']['batch_size'] = group['batch_size']
    return cfg


def write_group_configs(groups: list[dict]) -> list[Path]:
    GENERATED_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
    paths = []
    for group in groups:
        path = GENERATED_CONFIG_DIR / f"dyck_counter_task_a_{group['name']}.yaml"
        path.write_text(yaml.safe_dump(make_group_config(group), sort_keys=False), encoding='utf-8')
        paths.append(path)
    return paths


config_paths = write_group_configs(TASK_A_GROUPS) if WRITE_CONFIGS else [CONFIG_TEMPLATE]
config_paths

In [ ]:
def pipeline_cmd(config_path: Path, *, seed: int | None = None) -> list[str]:
    cmd = [
        sys.executable,
        str(ROOT / 'scripts' / 'run_pipeline.py'),
        '--config',
        str(config_path),
        '--model',
        'transformer',
        '--num-examples',
        str(NUM_EXAMPLES),
        '--extract-batch-size',
        str(EXTRACT_BATCH_SIZE),
        '--probe-max-rows',
        str(PROBE_MAX_ROWS),
        '--probe-max-classes',
        str(PROBE_MAX_CLASSES),
    ]
    if seed is not None:
        cmd += ['--seed', str(seed)]
    if DEVICE:
        cmd += ['--device', DEVICE]
    return cmd


commands = [pipeline_cmd(path) for path in config_paths]
commands[:3]

In [ ]:
if RUN_TRAINING:
    for cmd in commands:
        print('$', ' '.join(cmd))
        subprocess.run(cmd, cwd=ROOT, check=True)

## Metric Collection

这里收集三类输出：

1. training/eval behavior：整体 next-token accuracy 和只在 Dyck target 上计算的 accuracy。
2. hidden-state probes：每层 linear readout 预测 left、right、height 和 legal-next class。
3. follow-up probes：oracle forced/free split、output-head alignment、direct intervention、cross-condition transfer、noise-schedule readout 和分桶 diagnostics。


In [ ]:
def load_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding='utf-8'))


def collect_task_a_results(results_root: Path = RESULTS_ROOT) -> pd.DataFrame:
    rows = []
    for run_config in sorted(results_root.glob('dyck_counter_task_a_*/*_seed*/config.json')):
        run_dir = run_config.parent
        cfg = load_json(run_config)
        metrics_path = run_dir / 'metrics.json'
        probes_path = run_dir / 'probes' / 'layerwise_probe.csv'
        if not metrics_path.exists() or not probes_path.exists():
            continue
        metrics = load_json(metrics_path)['eval']
        probes = pd.read_csv(probes_path)
        best = probes.sort_values('height_r2', ascending=False).iloc[0].to_dict()
        task = cfg['task']
        rows.append({
            'setting': cfg['setting_name'],
            'model': cfg['model_name'],
            'seed': cfg['seed'],
            'seq_len': task['seq_len'],
            'total_length': task['total_length'],
            'repeat_prob': task['repeat_prob'],
            'num_noise_tokens': task['num_noise_tokens'],
            'loss': metrics['loss'],
            'accuracy': metrics['accuracy'],
            'dyck_accuracy': metrics['dyck_accuracy'],
            'best_layer': int(best['layer']),
            'height_r2': best.get('height_r2'),
            'height_mae': best.get('height_mae'),
            'left_r2': best.get('left_r2'),
            'right_r2': best.get('right_r2'),
            'gap_height_r2_minus_dyck_acc': best.get('height_r2') - metrics['dyck_accuracy'],
        })
    return pd.DataFrame(rows)


summary = collect_task_a_results()
summary

In [ ]:
if not summary.empty:
    display(summary.groupby(['seq_len', 'repeat_prob', 'num_noise_tokens'])[['dyck_accuracy', 'height_r2', 'gap_height_r2_minus_dyck_acc']].mean().reset_index())

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    summary.plot.scatter(x='dyck_accuracy', y='height_r2', c='seq_len', colormap='viridis', ax=axes[0])
    axes[0].set_title('Behavior vs hidden count probe')
    axes[0].set_xlabel('Dyck next-token accuracy')
    axes[0].set_ylabel('Best-layer height R2')

    summary.sort_values('seq_len').plot.bar(x='setting', y='gap_height_r2_minus_dyck_acc', ax=axes[1])
    axes[1].set_title('Probe-behavior gap')
    axes[1].set_ylabel('height R2 - dyck accuracy')
    axes[1].tick_params(axis='x', labelrotation=45)
    plt.tight_layout()

也可以试试第二篇文献里面写的，做 activation patching，有两个 sequence，一个正常 sequence，一个全是noise，把做 forward pass之后的 hidden state 都提取出来，然后把两边的 hidden state 做 patch，看看移过去有没有效果。可以有很多种 patching，以及 sweep 之后选择性 patch。

<!-- TASK_A_REWRITTEN_ANALYSIS -->
## What We Ran

这次不是比较不同 architecture，而是固定 Transformer 后做 length/noise sweep。所有 run 都完成了训练、hidden extraction、layerwise probe 和 extra probes。

| setting           | seq  | dyck_len | repeat_prob | noise_vocab | steps | batch_size | hidden_examples |
| ----------------- | ---- | -------- | ----------- | ----------- | ----- | ---------- | --------------- |
| tiny_extreme_long | 2000 | 20       | 0.010       | 64          | 15000 | 4          | 512             |
| clean_short       | 48   | 48       | 1.000       | 4           | 15000 | 128        | 4096            |
| noisy_short       | 120  | 48       | 0.500       | 16          | 15000 | 128        | 4096            |
| sparse_medium     | 400  | 200      | 0.250       | 16          | 15000 | 128        | 4096            |
| sparse_long       | 1000 | 400      | 0.250       | 16          | 15000 | 16         | 512             |
| extreme_long      | 2000 | 400      | 0.100       | 64          | 15000 | 4          | 512             |

长序列设置为了显存把 batch size 降低了；`tiny_extreme_long` 和 `extreme_long` 都是 2000 长上下文，但前者只有 20 个括号 token，用来分离“上下文很长”和“counter 本身很长”这两种压力。

<!-- TASK_A_REWRITTEN_ANALYSIS -->
## Accuracy Definitions

Main Result 表里有四个 behavior accuracy。它们都只关心 next-token 任务里“下一个 token 是 Dyck 括号”的位置；noise target 不进入这些 Dyck accuracy 的分母。

记 `pred_t = argmax(logits_t)`，`target_t = x_{t+1}`，`D = {t: target_t 是 open/close bracket}`，`correct_t = 1[pred_t = target_t]`。实现上 `D` 对应 label 表里的 `target_is_dyck_position=True`。

| metric | 分母 | 计算方法 | 含义 |
|---|---:|---|---|
| `Dyck acc` | training/eval batches 里的 `D` | `mean(correct_t)` | 模型在 Dyck target 上的原始 next-token accuracy。Main Result 表中这个值来自训练阶段保存的 eval metric；与 extra-probe 的 `all_dyck_targets model_acc` 定义相同，但采样 batch 可能不同。 |
| `oracle acc` | extra-probe sampled `D` | `mean(oracle_next_dyck_acc_t)` | 当前 stochastic Dyck generator 下的可预测上限。forced 位置 oracle=1，free 位置 oracle=0.5，所以它取决于 forced/free 的比例。 |
| `forced acc` | `D` 中 `forced_state != free` 的位置 | `mean(correct_t | forced)` | 当前 prefix 已经唯一决定下一个括号时，模型是否执行合法 Dyck 约束。`height<=0` 时必须 open；`remaining_opens<=0` 或 `remaining_dyck<=height` 时必须 close。 |
| `free acc` | `D` 中 `forced_state == free` 的位置 | `mean(correct_t | free)` | open 和 close 都合法、sampler 随机二选一时的模型 accuracy。这里 oracle baseline 是 0.5；接近 0.5 不表示模型没学会合法性，而是目标本身没有确定答案。 |

因此读 Main Result 时，核心不是单看 `Dyck acc` 高低，而是比较 `Dyck acc` 是否接近 `oracle acc`，以及失败主要落在 `forced acc` 还是 `free acc`。

<!-- TASK_A_REWRITTEN_ANALYSIS -->
## Main Result: Two Regimes After Adding `tiny_extreme_long`

原先五个 settings 呈现出一个比较干净的模式：Dyck next-token accuracy 看起来只有大约 0.53-0.62，但 oracle forced/free split 显示它们几乎贴着当前 stochastic generator 的 oracle ceiling。也就是说，这五组的低 accuracy 主要来自 free step 的 50/50 随机性，而不是合法性约束失败。

新加的 `tiny_extreme_long` 则是一个真正不同的 regime：它只有 20 个括号 token 混在 2000 长上下文里，hidden 里仍有可读 count signal（height R2 约 0.80，legal-next probe 约 0.97），但行为没有贴近 oracle。特别是 forced accuracy 只有约 0.26，free accuracy 近乎 0。这说明“上下文极长 + 有效监督极稀疏”会造成比 Markov free-step ceiling 更严重的输出失败。

| setting           | Dyck acc | oracle acc | forced acc | free acc | best height R2 | legal-next probe |
| ----------------- | -------- | ---------- | ---------- | -------- | -------------- | ---------------- |
| tiny_extreme_long | 0.113    | 0.680      | 0.263      | 0.005    | 0.804          | 0.970            |
| clean_short       | 0.603    | 0.597      | 1.000      | 0.501    | 0.966          | 1.000            |
| noisy_short       | 0.622    | 0.608      | 1.000      | 0.501    | 0.894          | 1.000            |
| sparse_medium     | 0.559    | 0.554      | 1.000      | 0.501    | 0.780          | 0.994            |
| sparse_long       | 0.528    | 0.539      | 0.998      | 0.501    | 0.848          | 0.982            |
| extreme_long      | 0.555    | 0.539      | 0.986      | 0.501    | 0.730          | 0.958            |

这张表的核心读法：`forced acc` 代表规则已决定下一步时模型能否执行 Dyck 约束；`free acc` 代表规则没有决定时模型能否猜中随机采样结果。除 `tiny_extreme_long` 外，free acc 稳定在 0.501 左右，正好说明这一部分没有可学习的确定目标；`tiny_extreme_long` 的 forced/free 都低，说明它不是 oracle ceiling 问题，而是稀疏信号下的行为读出失败。

<!-- TASK_A_SPARSE_SUPERVISION_ABLATION -->
## Sparse Supervision Ablation: Fixed `seq_len=2000`

这里不改 Dyck 生成规则，只固定 `seq_len=2000` 和 `noise_vocab=64`，扫已完成的 bracket token 数量：`[20, 24, 28, 32, 34, 36, 40, 44, 48, 56, 64, 80, 100, 200, 400]`。当前 balanced Dyck generator 要求括号 token 总数为偶数，所以用户提出的 33/35 没有直接运行；这里用 34 补在转折区间里。目的就是看 `tiny_extreme_long` 的失败到底来自 2000 长上下文本身，还是来自 Dyck target 在训练 loss 里太稀疏。

| brackets | density | Dyck acc | oracle | forced | free  | height R2 | legal-next |
| -------- | ------- | -------- | ------ | ------ | ----- | --------- | ---------- |
| 20       | 0.010   | 0.098    | 0.680  | 0.263  | 0.005 | 0.804     | 0.970      |
| 24       | 0.012   | 0.063    | 0.663  | 0.187  | 0.004 | 0.822     | 0.965      |
| 28       | 0.014   | 0.071    | 0.647  | 0.221  | 0.009 | 0.815     | 0.975      |
| 32       | 0.016   | 0.071    | 0.640  | 0.236  | 0.007 | 0.793     | 0.867      |
| 34       | 0.017   | 0.125    | 0.634  | 0.447  | 0.007 | 0.777     | 0.971      |
| 36       | 0.018   | 0.169    | 0.633  | 0.602  | 0.013 | 0.783     | 0.966      |
| 40       | 0.020   | 0.226    | 0.625  | 0.777  | 0.043 | 0.755     | 0.859      |
| 44       | 0.022   | 0.215    | 0.621  | 0.760  | 0.042 | 0.753     | 0.871      |
| 48       | 0.024   | 0.227    | 0.612  | 0.826  | 0.054 | 0.779     | 0.894      |
| 56       | 0.028   | 0.305    | 0.606  | 0.927  | 0.137 | 0.764     | 0.903      |
| 64       | 0.032   | 0.566    | 0.600  | 0.977  | 0.463 | 0.732     | 0.897      |
| 80       | 0.040   | 0.577    | 0.590  | 0.962  | 0.492 | 0.781     | 0.902      |
| 100      | 0.050   | 0.578    | 0.578  | 0.988  | 0.503 | 0.685     | 0.897      |
| 200      | 0.100   | 0.554    | 0.556  | 0.990  | 0.499 | 0.733     | 0.938      |
| 400      | 0.200   | 0.539    | 0.539  | 0.986  | 0.501 | 0.730     | 0.958      |

结果现在更具体：20、24、28、32 个括号时，forced accuracy 仍然只有约 0.19-0.26，说明模型没有稳定执行强制 open/close 规则；34 个括号时 forced accuracy 抬到约 0.45，是转折的前沿；36 个括号时跳到约 0.60，40-48 个括号已经到约 0.76-0.83。56 个括号时 forced accuracy 已经到约 0.93，说明 forced-rule readout 基本恢复。但 free accuracy 要到 64 个括号才从 0.14 跳到约 0.46，并在 80/100/200/400 附近稳定到约 0.49-0.50。因此当前 seed=0 下有两段变化：forced 规则执行的突变大致在 34-40 brackets，free/random target 行为接近 oracle 的突变大致在 56-64 brackets。到 80-100 个括号时，Dyck accuracy 已经接近 200/400 个括号的长上下文结果。与此同时 hidden probe 一直不低，尤其 legal-next probe 在所有密度下都很高。所以更合理的解释是：`tiny_extreme_long` 的主要瓶颈不是 2000 长度本身，而是 Dyck supervision 在 next-token loss 中过于稀疏，导致输出头没有稳定学会 bracket readout。

<!-- TASK_A_SPARSE_SUPERVISION_ABLATION -->
![sparse_supervision_ablation](../../figures/dyck_counter_sparse_supervision_ablation/sparse_supervision_ablation.png)

<!-- TASK_A_REWRITTEN_ANALYSIS -->
### Behavior / Probe Overview

![task_a_overview](../../figures/dyck_counter_task_a/task_a_overview.png)

![task_a_height_axis_projection](../../figures/dyck_counter_task_a/task_a_height_axis_projection.png)

这些图说明 hidden 里确实有可读的 count/height structure：clean/noisy short 最强，长序列和更稀疏噪声下 R2 下降但仍明显高于行为 accuracy。height-axis projection 也显示表征沿 count 方向有连续结构，而不是 probe 偶然捡到噪声。

<!-- TASK_A_REWRITTEN_ANALYSIS -->
## Probe 1: Oracle Forced/Free Split

这个 probe 把 Dyck target 分成两类：

- forced：由当前 prefix 和 balanced-parentheses 约束唯一决定下一步，oracle accuracy = 1。
- free：open 和 close 都合法，当前 sampler 随机采样，oracle accuracy = 0.5。

| setting           | model forced | model free | forced_oracle | free_oracle |
| ----------------- | ------------ | ---------- | ------------- | ----------- |
| tiny_extreme_long | 0.263        | 0.005      | 1.000         | 0.500       |
| clean_short       | 1.000        | 0.501      | 1.000         | 0.500       |
| noisy_short       | 1.000        | 0.501      | 1.000         | 0.500       |
| sparse_medium     | 1.000        | 0.501      | 1.000         | 0.500       |
| sparse_long       | 0.998        | 0.501      | 1.000         | 0.500       |
| extreme_long      | 0.986        | 0.501      | 1.000         | 0.500       |

结果分成两类：原先五组 forced 位置几乎全对、free 位置稳定在随机上限；`tiny_extreme_long` 则 forced 也明显失败。因此，原始 Dyck accuracy 不是一个纯粹的 algorithmic counting 指标；它既会混合“可由规则决定的合法性”和“生成器随机 coin flip”，也会在极稀疏长上下文中暴露真正的 behavior/readout failure。

<!-- TASK_A_REWRITTEN_ANALYSIS -->
![extra_probe_oracle_forced_free](../../figures/dyck_counter_task_a_extra_probes/extra_probe_oracle_forced_free.png)

右图还能解释为什么原先五组的整体 accuracy 会随着设置变化：不同长度/稀疏度下 forced/free 的比例不同。free rows 占大多数时，总体 accuracy 会自然靠近 0.5。`tiny_extreme_long` 额外说明了一点：当 Dyck token 在 2000 长上下文里过于稀疏时，模型甚至没有稳定学会 forced rows。

<!-- TASK_A_REWRITTEN_ANALYSIS -->
## Probes 2-3: Is The Counter Direction Directly Used By The Output Head?

这里问的是更强的因果问题：height 在线性 probe 中可读，并不自动说明 output head 正沿着同一个方向做 close-vs-open 决策。所以我们做了两步：先比较 final-layer height direction 和 output head 的 close-minus-open 向量，再沿 height direction 直接移动 final hidden，看 bracket probability 是否系统变化。

| setting | final_cos | axis-margin_corr | P(close/bracket)_at_delta_0 | margin_shift/axis_std |
| --- | --- | --- | --- | --- |
| tiny_extreme_long | -0.021 | 0.330 | 0.504 | -0.006 |
| clean_short | -0.065 | 0.344 | 0.496 | -0.003 |
| noisy_short | -0.015 | 0.376 | 0.496 | -0.002 |
| sparse_medium | -0.042 | 0.223 | 0.498 | -0.008 |
| sparse_long | -0.017 | 0.183 | 0.491 | -0.003 |
| extreme_long | -0.006 | 0.105 | 0.501 | -0.001 |

结果偏谨慎：cosine 基本接近 0 且略负；axis-margin correlation 为正，说明沿数据流形 height 和 close/open margin 有相关性；但 direct final-hidden intervention 几乎不改变 `P(close | bracket logits)`。这支持一个更细的说法：counter 在 hidden state 中可读，但 output head 并没有简单地把 probe 找到的 height 方向当作直接控制旋钮。

<!-- TASK_A_REWRITTEN_ANALYSIS -->
![extra_probe_output_head_causal](../../figures/dyck_counter_task_a_extra_probes/extra_probe_output_head_causal.png)

这个 intervention 只是 final hidden 上的 direct-logit intervention，不等价于在中间层改激活后重新 forward。下一节补的是更接近 forward computation 的 layer-wise activation patch。

<!-- TASK_A_REWRITTEN_ANALYSIS -->
## Ablation: Does Removing The Height Direction Matter?

为了检验 probe direction 是否是 output head 真正在用的方向，我们在 final hidden 上做 ablation：把每个 hidden state 沿 final-layer height probe direction 的投影移除，再重新过同一个 output head。同时脚本里还保存了 shuffle-height-axis 和 remove-random-direction 两个对照。

| setting           | delta acc all | delta acc forced | delta acc free | delta NLL all |
| ----------------- | ------------- | ---------------- | -------------- | ------------- |
| tiny_extreme_long | 0.0007        | 0.0019           | 0.0000         | -0.0009       |
| clean_short       | 0.0005        | 0.0000           | 0.0006         | -0.0000       |
| noisy_short       | 0.0001        | 0.0000           | 0.0001         | -0.0001       |
| sparse_medium     | -0.0002       | 0.0000           | -0.0003        | -0.0001       |
| sparse_long       | 0.0005        | 0.0000           | 0.0005         | -0.0000       |
| extreme_long      | -0.0002       | 0.0002           | -0.0002        | 0.0000        |

结果基本是负结果：移除 height direction 后，Dyck-target accuracy 和 NLL 几乎不变。这和前面的 output-head cosine/direct intervention 一致，说明当前 linear height probe 找到的方向主要是可读表征方向，而不是 final logits 的直接控制方向。所以这里不能把“height R2 高”直接解释成“输出头沿这个方向执行计数决策”。

<!-- TASK_A_REWRITTEN_ANALYSIS -->
![height_direction_ablation](../../figures/dyck_counter_task_a_ablation/height_direction_ablation.png)

<!-- TASK_A_ACTIVATION_PATCH -->
## Layer-Wise Activation Patch: Does The Counter Direction Control The Forward Pass?

前面的 direct intervention 只是在 final hidden 上改 logits 前的向量。这里补了更接近因果机制的 patch：在某一层的 sequence activation 上沿该层 height probe direction 加/减若干 axis std，然后继续跑后续 Transformer layers 和 output head。当前先对固定 `seq_len=2000` 的 sparse ladder 做小样本 smoke test；为了避免同一序列里多个 patch 互相污染，每条 sampled sequence 只选一个 Dyck target prefix。

| setting             | brackets | best layer | height slope | random slope |
| ------------------- | -------- | ---------- | ------------ | ------------ |
| tiny_extreme_long   | 20       | 0          | 0.0013       | -0.0010      |
| sparse_len2000_b48  | 48       | 2          | -0.0011      | 0.0030       |
| sparse_len2000_b200 | 200      | 0          | 0.0010       | -0.0010      |
| extreme_long        | 400      | 1          | 0.0008       | 0.0001       |

结果仍然是负向的：`P(close | bracket logits)` 对 height-direction patch 的斜率只有约 0.001 量级，和 random-direction control 同量级。这个小样本结果不能替代正式 multi-seed 大样本统计，但它没有支持“probe 找到的 counter direction 可以在 forward computation 中直接控制 open/close 决策”。更像是 counter 信息可读，但模型实际决策可能使用了分布式、非线性或不同坐标的特征。

<!-- TASK_A_ACTIVATION_PATCH -->
![layerwise_activation_patch](../../figures/dyck_counter_task_a_activation_patch/layerwise_activation_patch.png)

<!-- TASK_A_CROSS_MODEL_PATCH -->
## Cross-Model Activation Patch Pilot

这个 pilot 问一个更直接的问题：如果把结果较好的 sparse-ladder 模型的 activation patch 到结果较差的模型里，坏模型的 forced open/close 行为会不会恢复？这里所有 pair 都固定在 `seq_len=2000, noise_vocab=64`，输入 tokens 来自 recipient sampler；donor 坏模型和 recipient 好模型吃同一批 tokens，然后只在每条 sequence 选一个 Dyck-target prefix position 做 patch。

expanded pilot 设置：256 examples，batch size 4，pairs 为 `b100/b80/b64 -> b20/b34`，再加两个 reverse controls：`b20->b100`, `b34->b100`。主要看 forced split，因为 free split 的 oracle 本来只有 0.5。表中 `L2` 表示 patch recipient 的最后一层 activation 后直接过 recipient output head。

主表表头定义如下：

- `pair`: `donor->recipient`，左边模型提供 activation，右边模型接收 patch，并用右边模型的后续层/output head 评估。
- `recipient forced`: recipient 不做 patch 时，在 forced split 上的 baseline accuracy。
- `donor forced`: donor 在同一批 recipient-sampled tokens/targets 上的 forced accuracy，用来确认 donor 本身是否更好。
- `L2 full`: 在 recipient 的 layer 2 位置，把被评估 prefix 的整段 hidden vector 替换成 donor 同输入、同位置 activation 后的 forced accuracy。
- `delta full`: `L2 full - recipient forced`，正数表示 full-state patch 改善 recipient。
- `L2 shuffle`: 用同一 batch 内打乱后的 donor layer-2 full activation 做 patch，是 full-state distribution control。
- `L2 retrieval match`: 从 donor hidden bank 里按 `forced_state + height + position_bin` 匹配一个 activation，再替换 recipient layer-2 full activation。
- `L2 retrieval state-rand`: 从 donor hidden bank 里只按 `forced_state` 随机取 activation，是 retrieval 的弱匹配/分布对照。
- `L2 height-scalar`: 只把 donor 同输入 activation 在 height-probe direction 上的 scalar，映射并注入 recipient 的 height direction；不替换整个 hidden vector。
- `L2 retrieval height`: 和 `L2 height-scalar` 类似，但 donor scalar 来自 matched retrieval activation。
- `delta NLL full`: `target NLL(L2 full) - target NLL(recipient baseline)`；负数表示 full-state patch 让目标 token 概率变高。

| pair      | recipient forced | donor forced | L2 full | delta full | L2 shuffle | L2 retrieval match | L2 retrieval state-rand | L2 height-scalar | L2 retrieval height | delta NLL full |
| --------- | ---------------- | ------------ | ------- | ---------- | ---------- | ------------------ | ----------------------- | ---------------- | ------------------- | -------------- |
| b100->b20 | 0.302            | 0.570        | 0.895   | 0.593      | 0.663      | 0.849              | 0.860                   | 0.302            | 0.302               | -2.415         |
| b100->b34 | 0.417            | 0.639        | 0.944   | 0.528      | 0.556      | 0.819              | 0.931                   | 0.417            | 0.403               | -1.542         |
| b20->b100 | 1.000            | 0.000        | 0.167   | -0.833     | 0.095      | 0.952              | 0.833                   | 1.000            | 1.000               | 2.039          |
| b34->b100 | 0.951            | 0.049        | 0.732   | -0.220     | 0.366      | 0.951              | 0.976                   | 0.951            | 0.951               | 1.167          |
| b64->b20  | 0.205            | 0.458        | 0.627   | 0.422      | 0.518      | 0.651              | 0.880                   | 0.205            | 0.205               | -1.543         |
| b64->b34  | 0.475            | 0.593        | 0.864   | 0.390      | 0.542      | 0.780              | 0.881                   | 0.458            | 0.458               | -0.678         |
| b80->b20  | 0.194            | 0.510        | 0.520   | 0.327      | 0.520      | 0.582              | 0.643                   | 0.194            | 0.194               | -1.246         |
| b80->b34  | 0.479            | 0.521        | 0.577   | 0.099      | 0.535      | 0.676              | 0.606                   | 0.451            | 0.465               | -0.607         |

结果有三个清楚信号。第一，最后层 `full_state_replace` 对 good->bad pair 仍有强提升：`b100->b20` forced acc 从约 0.30 到 0.90，`b100->b34` 从约 0.42 到 0.94，`b64->b34` 从约 0.47 到 0.86。反向 `b20->b100` 则把 forced acc 从约 1.00 打到 0.17，说明最后层 hidden state 确实携带行为相关信息。

第二，donor activation retrieval 也能改善坏 recipient。`retrieval_full_match` 用 donor bank 中按 `forced_state + height + position_bin` 匹配的 activation；它在 `b100->b20`, `b100->b34`, `b64->b20`, `b64->b34`, `b80->b20`, `b80->b34` 上都提升 forced accuracy。但 `retrieval_full_state_random` 经常和 match 一样强甚至更强，说明主要效应目前更像是把 recipient final hidden 推到 donor-like activation distribution，而不是已经证明了逐样本语义匹配的 donor counter state。

第三，`height_scalar_replace` 和 `retrieval_height_match` 基本没有改善，和前面的 height-direction intervention / ablation 结论一致：linear height probe 的 scalar 本身不是一个足够的 causal control。所以目前的解释应当是：完整 hidden state 的分布/方向携带行为相关信息，但单一 linear counter scalar 不是输出决策的控制旋钮。下一步如果要更严谨，应该把 retrieval 从粗粒度 key matching 改成 nearest-neighbor matching，并报告 match-vs-state-random 的置信区间。

下面的 top-effect 表列出 forced split 上 `delta_accuracy_vs_recipient` 最大的若干 patch 条件：`layer` 是被 patch 的 recipient 层，`mode` 是 patch 类型，`delta_nll_vs_recipient` 为负表示目标 NLL 下降，`gap_closed` 是 patch 改善量相对 donor-recipient baseline 差距的比例；`gap_closed > 1` 说明 patch 后甚至超过了 donor 在同一批样本上的 forced accuracy。

| donor | recipient | layer | mode                        | delta_accuracy_vs_recipient | delta_nll_vs_recipient | gap_closed |
| ----- | --------- | ----- | --------------------------- | --------------------------- | ---------------------- | ---------- |
| b64   | b20       | 2     | retrieval_full_state_random | 0.6747                      | -0.7854                | 2.6667     |
| b100  | b20       | 2     | full_state_replace          | 0.5930                      | -2.4154                | 2.2174     |
| b100  | b20       | 2     | retrieval_full_state_random | 0.5581                      | -1.0081                | 2.0870     |
| b100  | b20       | 2     | retrieval_full_match        | 0.5465                      | -0.9822                | 2.0435     |
| b100  | b34       | 2     | full_state_replace          | 0.5278                      | -1.5418                | 2.3750     |
| b100  | b34       | 2     | retrieval_full_state_random | 0.5139                      | -0.8683                | 2.3125     |
| b80   | b20       | 2     | retrieval_full_state_random | 0.4490                      | -0.5329                | 1.4194     |
| b64   | b20       | 2     | retrieval_full_match        | 0.4458                      | -0.5005                | 1.7619     |

<!-- TASK_A_CROSS_MODEL_PATCH -->
![cross_model_patch_pilot](../../figures/dyck_counter_task_a_cross_model_patch/cross_model_patch_pilot.png)

<!-- TASK_A_REWRITTEN_ANALYSIS -->
## Probes 4-6: Transfer, Noise Schedule, And Binned Diagnostics

这组 probe 回答三个问题：不同设置学到的 counter 坐标系是否共享；hidden state 是否线性编码“下一个 token 是不是 Dyck”；错误集中在哪些 height/position 区域。

| setting           | self-transfer R2 | next-is-dyck over baseline | next-symbol over baseline |
| ----------------- | ---------------- | -------------------------- | ------------------------- |
| tiny_extreme_long | 0.851            | -0.000                     | 0.000                     |
| clean_short       | 0.980            | N/A                        | 0.097                     |
| noisy_short       | 0.956            | 0.004                      | 0.003                     |
| sparse_medium     | 0.907            | 0.018                      | -0.004                    |
| sparse_long       | 0.908            | -0.006                     | -0.005                    |
| extreme_long      | 0.827            | -0.003                     | -0.003                    |

cross-condition transfer 的结论是：每个 run 自己的 height probe 都能工作，但跨设置迁移大多很差，只有 `sparse_medium` 和 `sparse_long` 之间有一点正迁移。这说明 counter geometry 目前更像是每个训练条件内部的局部坐标，而不是已经对齐成一个通用坐标系。

noise-schedule probe 加入 majority baseline 后，next-is-dyck / next-symbol 的优势基本接近 0。`clean_short` 的 next-symbol 有一个小例外，因为它没有 noise 插入，target type 退化成 bracket symbol readout；其余 noisy/sparse 设置没有显示出稳定的线性 schedule signal。

binned diagnostics 则把行为现象定位得更具体：原先五组里，height=0 的 forced-open 区域几乎全对，height>0 后大部分 free 区域回到约 0.5；`tiny_extreme_long` 的分桶则显示稀疏长上下文下 forced 区域也会失败。序列后段 accuracy 上升时，通常来自必须 close 的 forced 区域变多。

<!-- TASK_A_REWRITTEN_ANALYSIS -->
![extra_probe_transfer_noise_bins](../../figures/dyck_counter_task_a_extra_probes/extra_probe_transfer_noise_bins.png)

<!-- TASK_A_REWRITTEN_ANALYSIS -->
## Overall Interpretation

这次 Task A 的结论可以压缩成三句话。

第一，Transformer 确实学到了 Dyck prefix counter 的 hidden-state representation。`left/right/height` 都能被线性 probe 读出，legal-next class 也几乎可读。

第二，当前原始 Dyck next-token accuracy 不是衡量 counter 能力的好单一指标。原先五组里，Markov-style sampler 的 free step 本来就是随机的，模型整体 accuracy 几乎等于 oracle forced/free baseline；但 `tiny_extreme_long` 显示，在 2000 长上下文里只放 20 个括号时，模型连 forced 位置也不能稳定输出。稀疏监督 ablation 进一步说明：20-32 brackets 仍失败，34 brackets 是 forced 恢复的前沿，36-48 brackets 进入快速恢复区间，56 brackets 后 forced 规则基本能执行；但 free/random target 直到 64 brackets 才接近 oracle baseline。所以主要瓶颈是 Dyck target 在训练中太稀疏，当前 seed=0 下更像有两段转折：forced readout 在 34-40，整体 Dyck/free 行为在 56-64。

第三，虽然 counter 可读，但它在不同 length/noise 条件下没有形成统一坐标系；final output head 没有简单沿 probe height direction 使用它，移除该方向的 ablation 也几乎不改变行为。layer-wise height-direction patch smoke test 也没有看到 height direction 对 `P(close)` 的系统控制。cross-model patch 则补充了另一面：好模型的最后层 full activation patch 到坏模型后，forced 行为可以明显改善；donor-bank retrieval 也能改善坏 recipient。但只替换 height scalar 基本无效，而且 state-random retrieval 经常和 matched retrieval 一样强。因此当前结果支持“行为相关信息存在于更完整的 hidden state / activation distribution 中”，但还不能支持“线性 height probe scalar 就是可直接因果控制的通用计数旋钮”。